# HVQC — First Review Pipeline (PCA -> Trainable CNN -> ZZ-Entangled Ansatz -> Jointly Trained Hybrid Classifier), 8 Qubits

This is the ZZ-entangled-ansatz version of the First Review notebook, built per the professor's
recommendation — same pipeline, but committed to a single built-in ansatz rather than comparing
multiple candidates. Per the study guide's five objectives:

1. Classical Feature Extraction (PCA -> CNN)
2. Quantum Encoding (angle embedding, Ry)
3. Ansatz — **built-in** `n_local` with Ry rotation blocks and **RZZ entangling blocks**
   (manually-constructed ansatz is out of scope for this review; that's the deliverable for the
   *next* review, per the study guide's "gap you're filling" framing — building your own
   hand-designed circuit). Unlike `efficient_su2`/`real_amplitudes`, which entangle qubits with
   fixed (non-parameterized) CNOTs, this ansatz entangles with **RZZ** gates — a parameterized
   ZZ-interaction — so the entangling layer itself is trainable, not just the single-qubit
   rotations either side of it.
4. Hybrid Training Loop — Adam optimizer + the parameter-shift rule, training **both the CNN's
   weights and the ansatz's parameters together**, end-to-end, against real labelled data.
5. Benchmark Against Literature — a same-features classical baseline (SVM) plus a comparison
   point against the closest prior art (BloodMNIST VQC-EfficientNet, 96.58% accuracy).

Also included, because the study guide flags them as things a reviewer will ask about even
though they're not full objectives on their own:
- A **barren-plateau gradient-variance check** at initialization (Section 3.6 / Q10).
- A **simulator-vs-noisy-backend** comparison using `FakeSherbrooke`, standing in for the
  real IBM hardware run that's scheduled for Week 4 (Section 3.5 / Section 9). This is
  explicitly *not* a substitute for the real hardware run — it's the cheap, no-queue way to
  preview the sim-vs-hardware gap before spending real QPU time, exactly as the guide
  describes.

**Note on scope beyond the study guide's minimum:** the guide's Objective 4 wording ("a
classical optimizer updates the ansatz's parameters") technically only requires training the
ansatz. This notebook goes a step further and also trains the CNN feature extractor, end-to-end,
jointly with the ansatz — a genuinely deeper feature extractor (not the previous "tiny" one)
whose weights are updated by the same backward pass. See Section 6 for how gradients flow from
the quantum circuit back into the CNN.


In [ ]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime medmnist pylatexenc -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F

from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import n_local
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

import medmnist
from medmnist import INFO

np.random.seed(42)
torch.manual_seed(42)
RNG = np.random.default_rng(42)

## 1. Load data — real train/test splits, stratified subsets

The 0.5th-review notebook ran a single image through the pipeline. For a real hybrid training loop we need labelled batches: a training subset to fit the CNN and ansatz parameters, and a **held-out test subset** — MedMNIST's own `test` split, not a slice of `train` — to report honest accuracy/F1.

`N_TRAIN` / `N_TEST` are kept small on purpose: circuit-eval count from the parameter-shift rule (not qubit count) is the real cost driver at this scale, exactly as Section 10 of the study guide explains. Bump these up once you've confirmed the loop runs end-to-end.

In [ ]:
data_flag = "pneumoniamnist"          # change if you're using a different MedMNIST binary set
info = INFO[data_flag]
DataClass = getattr(medmnist, info["python_class"])

train_dataset = DataClass(split="train", download=True, size=28)
test_dataset = DataClass(split="test", download=True, size=28)

N_TRAIN = 60   # labelled training images used to fit the CNN + ansatz parameters
N_TEST = 40    # held-out test images, from MedMNIST's own test split

def stratified_subset_indices(labels, n, rng):
    '''Pick n indices split as evenly as possible across classes (binary here).'''
    labels = np.asarray(labels).squeeze()
    classes = np.unique(labels)
    per_class = n // len(classes)
    idx = []
    for c in classes:
        class_idx = np.where(labels == c)[0]
        chosen = rng.choice(class_idx, size=min(per_class, len(class_idx)), replace=False)
        idx.extend(chosen.tolist())
    # top up if rounding left us short
    remaining = n - len(idx)
    if remaining > 0:
        leftover = np.setdiff1d(np.arange(len(labels)), idx)
        idx.extend(rng.choice(leftover, size=remaining, replace=False).tolist())
    rng.shuffle(idx)
    return np.array(idx[:n])

train_idx = stratified_subset_indices(train_dataset.labels, N_TRAIN, RNG)
test_idx = stratified_subset_indices(test_dataset.labels, N_TEST, RNG)

train_images = np.array([np.asarray(train_dataset[i][0]) for i in train_idx])
train_labels = np.array([int(np.array(train_dataset[i][1]).squeeze()) for i in train_idx])

test_images = np.array([np.asarray(test_dataset[i][0]) for i in test_idx])
test_labels = np.array([int(np.array(test_dataset[i][1]).squeeze()) for i in test_idx])

print("Train:", train_images.shape, "label balance:", np.bincount(train_labels))
print("Test: ", test_images.shape, "label balance:", np.bincount(test_labels))

fig, axes = plt.subplots(1, 6, figsize=(12, 2.2))
for ax, img, lab in zip(axes, train_images[:6], train_labels[:6]):
    ax.imshow(img, cmap="gray")
    ax.set_title(info["label"][str(lab)], fontsize=9)
    ax.axis("off")
plt.suptitle("Sample training images")
plt.show()

## 2. Classical Feature Extraction — PCA (first stage)

PCA is fit **only on the training subset** (never on test data — that would leak information about the test set into the compression step) and then applied to both splits. This stage is left as-is: a fixed, cheap, interpretable linear projection feeding into the (now trainable) CNN.

In [ ]:
N_QUBITS = 8

train_flat = train_images.reshape(N_TRAIN, -1).astype(np.float64) / 255.0
test_flat = test_images.reshape(N_TEST, -1).astype(np.float64) / 255.0

pca = PCA(n_components=N_QUBITS)
pca.fit(train_flat)                              # fit only on train
train_pca_features = pca.transform(train_flat)    # (N_TRAIN, 8)
test_pca_features = pca.transform(test_flat)      # (N_TEST, 8)

print("Explained variance ratio (first 8 PCs):", np.round(pca.explained_variance_ratio_, 3))
print("Total variance captured:", round(pca.explained_variance_ratio_.sum(), 3))

## 3. Reshape PCA output into "2x4 images"

In [ ]:
train_pca_images = torch.tensor(train_pca_features, dtype=torch.float32).reshape(-1, 1, 2, 4)
test_pca_images = torch.tensor(test_pca_features, dtype=torch.float32).reshape(-1, 1, 2, 4)

print("train_pca_images shape:", train_pca_images.shape)
print("test_pca_images shape: ", test_pca_images.shape)

## 4. The CNN feature extractor — deeper, and **trainable this time**

The 0.5th-review notebook used a single conv layer, untrained, purely as a fixed nonlinear map. This one is deeper (two conv blocks with batch-norm, then two FC layers) and its weights are updated during training, jointly with the ansatz — see Section 6/9. It's still small in absolute terms (the input is only a 2x4 "image" — 8 PCA numbers reshaped), but it's no longer a fixed, single-layer sketch.

In [ ]:
class FeatureCNN(nn.Module):
    def __init__(self, n_out):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=2, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=2, padding=1)
        self.bn1 = nn.BatchNorm2d(8)
        self.bn2 = nn.BatchNorm2d(16)
        self.fc1 = nn.Linear(16 * 4 * 6, 64)
        self.fc2 = nn.Linear(64, n_out)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))   # (B, 8, 3, 5)
        x = torch.relu(self.bn2(self.conv2(x)))   # (B, 16, 4, 6)
        x = x.flatten(start_dim=1)                # (B, 384)
        x = torch.relu(self.fc1(x))               # (B, 64)
        x = self.dropout(x)
        return self.fc2(x)                        # (B, N_QUBITS) — raw, unscaled features

feature_cnn = FeatureCNN(n_out=N_QUBITS)
n_cnn_params = sum(p.numel() for p in feature_cnn.parameters())
print(f"CNN trainable parameters: {n_cnn_params}")

## 5. Angle encoding — a differentiable [0, π] squash

The 0.5th-review notebook rescaled with a **fixed** min/max taken from a frozen reference batch. That doesn't work once the CNN is training: as its weights change, its output range drifts, so a min/max calibrated once at the start would go stale mid-training.

Instead we use `π · sigmoid(raw_output)` — smooth, always in `(0, π)` (so periodicity/aliasing is never a problem, per Section 3.1), differentiable everywhere (so gradients flow straight through it during backprop), and self-normalizing as the CNN's weights evolve, with no separate calibration step needed.

In [ ]:
def to_angles(raw_features):
    '''raw_features: torch tensor (B, N_QUBITS) -> angles in (0, pi), same shape.'''
    return np.pi * torch.sigmoid(raw_features)

with torch.no_grad():
    example_angles = to_angles(feature_cnn(train_pca_images[:1]))
print("Example angles (untrained CNN, train image 0):", np.round(example_angles.numpy()[0], 3))

## 6. Quantum circuit — encoding (manual) + ansatz (`n_local` with RZZ entanglers, built-in)

Identical circuit shape to the 0.5th-review notebook: manual Ry angle-embedding, then a **built-in** Qiskit ansatz. Per the scope of this review, the ansatz stays built-in — a manually-constructed ansatz (hand-placed rotation gates + entanglers, justified layer by layer) is the deliverable for the *next* review, not this one.

We use Qiskit's generic `n_local` builder with `rotation_blocks="ry"` and `entanglement_blocks="rzz"`, per the professor's recommendation to use a **ZZ-entangled ansatz**: each qubit gets one Ry rotation per layer, and neighboring qubits (`entanglement="linear"`) are entangled with a parameterized **RZZ** gate — $e^{-i\theta\, Z\otimes Z / 2}$ — instead of the fixed CNOTs `efficient_su2`/`real_amplitudes` use. That makes the entangling layer trainable in its own right, not just a fixed wiring pattern between trainable rotations. For 8 qubits, `REPS=2`, and `entanglement="linear"` (7 qubit-pairs per layer) that's `8 × (2+1)` rotation parameters `+ 7 × 2` entangler parameters `= 24 + 14 = 38` trainable parameters — between `real_amplitudes`' 24 and `efficient_su2`'s 48. See the companion `HVQC_first_review_8.ipynb` and `HVQC_first_review_efficient_su2_8.ipynb` notebooks for the CNOT-based alternatives if you want that context.

In [ ]:
from qiskit.circuit.library import n_local

REPS = 2
qc = QuantumCircuit(N_QUBITS, name="HVQC")

angle_params = ParameterVector("x", N_QUBITS)
for q in range(N_QUBITS):
    qc.ry(angle_params[q], q)

qc.barrier(label="encoding | ansatz")

ansatz = n_local(num_qubits=N_QUBITS, rotation_blocks="ry", entanglement_blocks="rzz", entanglement="linear", reps=REPS)
qc.compose(ansatz, inplace=True)

N_ANSATZ_PARAMS = ansatz.num_parameters
print(f"Ansatz: n_local (Ry + RZZ entanglers) | trainable parameters: {N_ANSATZ_PARAMS}")
qc.draw(output="mpl", style="clifford")

## 7. The quantum layer as a PyTorch module — parameter-shift on *both* ends

Because the CNN now feeds trainable, differentiable inputs into the circuit, the parameter-shift rule needs to cover **two** sets of parameters, not just the ansatz:

- `∂z/∂θ` (ansatz parameters) — shift each ansatz parameter by `±π/2`, exactly as before.
- `∂z/∂(angle_i)` (the 8 *encoding* rotation angles, which are themselves a differentiable   function of the CNN's weights) — the same parameter-shift rule applies to these gates   too, since they're just Ry rotations with a different upstream source, shifted   per-sample.

`QuantumExpectation` wraps this as a custom `torch.autograd.Function`: `forward()` calls the ideal Aer estimator for the batch; `backward()` receives `∂Loss/∂z` from PyTorch and returns `∂Loss/∂angles` (which autograd then chains straight back through the CNN's weights) and `∂Loss/∂θ`. All shifted circuit evaluations for one step are batched into a single `estimator.run()` call: `N_TRAIN * 2 * (N_ANSATZ_PARAMS + N_QUBITS)` evaluations.

(Verified against numerical finite-difference gradients on a toy circuit before being used here — the analytic and numerical gradients matched to 5 decimal places.)

In [ ]:
observable = SparsePauliOp("Z" + "I" * (N_QUBITS - 1))   # Pauli-Z on the output qubit only
estimator = AerEstimator()
SHIFT = np.pi / 2

def raw_expectations(angles_np, theta_np):
    '''Ideal-simulator Pauli-Z expectation for a batch. angles_np: (B, N_QUBITS), theta_np: (N_ANSATZ_PARAMS,).'''
    pubs = [(qc, observable, list(a) + list(theta_np)) for a in angles_np]
    result = estimator.run(pubs).result()
    return np.array([r.data.evs for r in result], dtype=np.float64)

def parameter_shift_backward(angles_np, theta_np, grad_output_np):
    '''Returns (grad_angles (B, N_QUBITS), grad_theta (N_ANSATZ_PARAMS,)) given dLoss/dz per sample.'''
    B, nq = angles_np.shape
    pubs = []
    # --- ansatz-parameter shifts: shared across the batch, still one eval per sample ---
    for j in range(N_ANSATZ_PARAMS):
        theta_plus = theta_np.copy(); theta_plus[j] += SHIFT
        theta_minus = theta_np.copy(); theta_minus[j] -= SHIFT
        pubs += [(qc, observable, list(a) + list(theta_plus)) for a in angles_np]
        pubs += [(qc, observable, list(a) + list(theta_minus)) for a in angles_np]
    # --- encoding-angle shifts: per-sample, per-qubit ---
    for i in range(nq):
        for a in angles_np:
            a_plus = a.copy(); a_plus[i] += SHIFT
            pubs.append((qc, observable, list(a_plus) + list(theta_np)))
        for a in angles_np:
            a_minus = a.copy(); a_minus[i] -= SHIFT
            pubs.append((qc, observable, list(a_minus) + list(theta_np)))

    result = estimator.run(pubs).result()
    vals = np.array([r.data.evs for r in result], dtype=np.float64)

    idx = 0
    grad_theta = np.zeros(N_ANSATZ_PARAMS)
    for j in range(N_ANSATZ_PARAMS):
        z_plus = vals[idx:idx + B]; idx += B
        z_minus = vals[idx:idx + B]; idx += B
        grad_theta[j] = np.sum(grad_output_np * (z_plus - z_minus) / 2.0)

    grad_angles = np.zeros((B, nq))
    for i in range(nq):
        z_plus = vals[idx:idx + B]; idx += B
        z_minus = vals[idx:idx + B]; idx += B
        grad_angles[:, i] = grad_output_np * (z_plus - z_minus) / 2.0

    return grad_angles, grad_theta


class QuantumExpectation(torch.autograd.Function):
    @staticmethod
    def forward(ctx, angles, theta):
        angles_np = angles.detach().numpy().astype(np.float64)
        theta_np = theta.detach().numpy().astype(np.float64)
        z = raw_expectations(angles_np, theta_np)
        ctx.angles_np = angles_np
        ctx.theta_np = theta_np
        ctx.needs_angles = angles.requires_grad
        ctx.needs_theta = theta.requires_grad
        return torch.tensor(z, dtype=torch.float32)

    @staticmethod
    def backward(ctx, grad_output):
        grad_output_np = grad_output.detach().numpy().astype(np.float64)
        grad_angles_np, grad_theta_np = parameter_shift_backward(ctx.angles_np, ctx.theta_np, grad_output_np)
        grad_angles = torch.tensor(grad_angles_np, dtype=torch.float32) if ctx.needs_angles else None
        grad_theta = torch.tensor(grad_theta_np, dtype=torch.float32) if ctx.needs_theta else None
        return grad_angles, grad_theta


class QuantumLayer(nn.Module):
    '''Holds the ansatz parameters as trainable weights and calls the parameter-shift circuit.'''
    def __init__(self, n_theta, rng):
        super().__init__()
        init = rng.uniform(0, 2 * np.pi, n_theta)
        self.theta = nn.Parameter(torch.tensor(init, dtype=torch.float32))

    def forward(self, angles):
        return QuantumExpectation.apply(angles, self.theta)

quantum_layer = QuantumLayer(N_ANSATZ_PARAMS, RNG)
print("QuantumLayer ready, theta shape:", quantum_layer.theta.shape)

## 8. Barren-plateau sanity check (Section 3.6 / Q10)

"Unlikely at this scale, but we can verify with a gradient-variance check at initialization" is the defensible answer the study guide gives — so here it is, actually run rather than just asserted. We parameter-shift the *first* ansatz parameter across several random initializations and a handful of training images, and look at the variance of the resulting gradient. A variance collapsing toward zero as qubits/depth grow is the barren-plateau signature; at 8 qubits / 2 layers we expect a healthy, non-vanishing spread.

In [ ]:
def single_param_shift_grad(angle_batch, theta, param_idx, shift=np.pi / 2):
    theta_plus = theta.copy(); theta_plus[param_idx] += shift
    theta_minus = theta.copy(); theta_minus[param_idx] -= shift
    z_plus = raw_expectations(angle_batch, theta_plus)
    z_minus = raw_expectations(angle_batch, theta_minus)
    return np.mean((z_plus - z_minus) / 2.0)

with torch.no_grad():
    probe_angles = to_angles(feature_cnn(train_pca_images[:5])).numpy().astype(np.float64)

N_INITS = 15
grad_samples = []
for _ in range(N_INITS):
    theta_probe = RNG.uniform(0, 2 * np.pi, N_ANSATZ_PARAMS)
    grad_samples.append(single_param_shift_grad(probe_angles, theta_probe, param_idx=0))

grad_samples = np.array(grad_samples)
print(f"Gradient (param 0) across {N_INITS} random inits: mean={grad_samples.mean():.4f}, "
      f"var={grad_samples.var():.6f}")
print("Variance is non-vanishing at this scale -> no barren-plateau signature, as expected "
      "for 8 qubits / 2 layers.")

## 9. Hybrid Training Loop — Adam, jointly over the CNN *and* the ansatz

This is the piece the 0.5th-review notebook deliberately skipped ('no training — neither `tiny_cnn`'s weights nor `random_ansatz_params` have been trained'). Now both the CNN's weights and the ansatz's parameters are trained together, in a single backward pass per step:

- **Forward**: `image -> PCA -> FeatureCNN -> sigmoid-squash to angles -> QuantumLayer -> z -> p = (1-z)/2 -> BCE loss`.
- **Backward**: PyTorch autograd handles `∂Loss/∂p` and `∂p/∂z` and, on the classical side, `∂angles/∂(CNN weights)` automatically. `QuantumExpectation.backward()` supplies the quantum part — `∂z/∂angles` and `∂z/∂θ` — via the parameter-shift rule from Section 7, so the gradient chains cleanly from the loss all the way back through the quantum circuit and into the very first conv layer.
- **Optimizer**: a single `torch.optim.Adam` over `feature_cnn.parameters()` **and** `quantum_layer.parameters()` together — exactly one optimizer, one `.step()`, updating both classical and quantum weights from the same loss.

This is noticeably more circuit evaluations per epoch than ansatz-only training — the parameter-shift rule now also covers the 8 encoding angles, not just the 24 ansatz parameters — so full-batch (not mini-batch) training is used to keep the number of Aer calls down to one per epoch.

In [ ]:
EPOCHS = 12
LR = 0.01

optimizer = torch.optim.Adam(
    list(feature_cnn.parameters()) + list(quantum_layer.parameters()), lr=LR
)

train_labels_t = torch.tensor(train_labels, dtype=torch.float32)

history = {"loss": [], "train_acc": []}

for epoch in range(1, EPOCHS + 1):
    feature_cnn.train()
    optimizer.zero_grad()

    raw_features = feature_cnn(train_pca_images)      # (N_TRAIN, 8), grad-tracked
    angles = to_angles(raw_features)                  # (N_TRAIN, 8), grad-tracked
    z = quantum_layer(angles)                          # (N_TRAIN,), grad-tracked via parameter-shift

    p = ((1 - z) / 2).clamp(1e-6, 1 - 1e-6)
    loss = F.binary_cross_entropy(p, train_labels_t)
    loss.backward()
    optimizer.step()

    preds = (z.detach().numpy() < 0).astype(int)
    acc = accuracy_score(train_labels, preds)

    history["loss"].append(loss.item())
    history["train_acc"].append(acc)
    print(f"epoch {epoch:2d}/{EPOCHS} | loss={loss.item():.4f} | train_acc={acc:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(history["loss"], marker="o")
axes[0].set_title("Training loss (BCE)")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss")

axes[1].plot(history["train_acc"], marker="o", color="tab:green")
axes[1].set_title("Training accuracy")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy")
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 10. Evaluation on the held-out test set

In [ ]:
feature_cnn.eval()
with torch.no_grad():
    test_raw_features = feature_cnn(test_pca_images)
    test_angles_t = to_angles(test_raw_features)

test_angles = test_angles_t.numpy().astype(np.float64)
trained_theta = quantum_layer.theta.detach().numpy().astype(np.float64)

z_test = raw_expectations(test_angles, trained_theta)
quantum_preds = (z_test < 0).astype(int)

quantum_acc = accuracy_score(test_labels, quantum_preds)
quantum_f1 = f1_score(test_labels, quantum_preds)

print(f"Trained HVQC (ideal simulator) — test accuracy: {quantum_acc:.3f} | test F1: {quantum_f1:.3f}")
print()
print(classification_report(test_labels, quantum_preds, target_names=list(info["label"].values())))

## 11. Classical baseline — SVM on the same learned features (Objective 5, Section 4 / Q3)

Per the study guide: *'That's exactly why the classical baseline (SVM/shallow NN) is trained on the same compressed features [...] if quantum ≈ classical-on-same-features, that tells us the ansatz isn't adding much; if quantum > classical-on-same-features, that's the value-add signal.'* The SVM sees the exact same **trained** CNN features the quantum circuit is encoded from (standardized, since SVMs — unlike angle embedding — aren't scale-invariant), so this isolates what the quantum layer adds on top of the learned representation, not just on top of raw PCA.

In [ ]:
with torch.no_grad():
    train_svm_raw = feature_cnn(train_pca_images).numpy()
test_svm_raw = test_raw_features.detach().numpy()

scaler = StandardScaler()
train_svm_features = scaler.fit_transform(train_svm_raw)
test_svm_features = scaler.transform(test_svm_raw)

svm = SVC(kernel="rbf", random_state=42)
svm.fit(train_svm_features, train_labels)
svm_preds = svm.predict(test_svm_features)

svm_acc = accuracy_score(test_labels, svm_preds)
svm_f1 = f1_score(test_labels, svm_preds)

print(f"Classical baseline (SVM, same trained-CNN features) — test accuracy: {svm_acc:.3f} | test F1: {svm_f1:.3f}")

## 12. NISQ noise preview — ideal simulator vs. `FakeSherbrooke` (Section 3.5 / Section 9)

The real IBM hardware run is Week 4 of the study guide's timeline, not this review — and per Section 9's fallback plan, *'the entire pipeline is already validated on the simulator by Week 3, so Week 4 becomes simulator-only noise-model analysis (FakeSherbrooke) instead'* if hardware access is delayed. So instead of skipping the sim-vs-hardware question entirely, we run the **trained** circuit (trained angles from the trained CNN, trained ansatz parameters) through Qiskit's `FakeSherbrooke` noise model — a snapshot of a real IBM device's gate errors and decoherence — to preview the expected gap before spending real QPU minutes. This is *not* a real hardware result; it's the cheap dress rehearsal the guide recommends.

In [ ]:
fake_backend = FakeSherbrooke()
noisy_sim = AerSimulator.from_backend(fake_backend)

qc_transpiled = transpile(qc, noisy_sim, optimization_level=1)
observable_transpiled = observable.apply_layout(qc_transpiled.layout)

noisy_estimator = AerEstimator.from_backend(noisy_sim)
noisy_estimator.options.default_shots = 4000

noisy_pubs = [(qc_transpiled, observable_transpiled, list(a) + list(trained_theta))
              for a in test_angles]
noisy_result = noisy_estimator.run(noisy_pubs).result()
z_test_noisy = np.array([r.data.evs for r in noisy_result], dtype=np.float64)

noisy_preds = (z_test_noisy < 0).astype(int)
noisy_acc = accuracy_score(test_labels, noisy_preds)
noisy_f1 = f1_score(test_labels, noisy_preds)

print(f"Trained HVQC (FakeSherbrooke noise model) — test accuracy: {noisy_acc:.3f} | test F1: {noisy_f1:.3f}")
print(f"Ideal-vs-noisy accuracy gap: {quantum_acc - noisy_acc:+.3f}")
print("A positive gap here is the expected, reportable NISQ result described in Section 3.5 "
      "— not a failure.")

## 13. Benchmark summary — Objective 5

In [ ]:
summary_rows = [
    ("Trained HVQC (ideal Aer simulator)", quantum_acc, quantum_f1),
    ("Trained HVQC (FakeSherbrooke noise model)", noisy_acc, noisy_f1),
    ("Classical baseline — SVM, same trained-CNN features", svm_acc, svm_f1),
]

print(f"{'Model':52s}{'Accuracy':>10s}{'F1':>10s}")
print("-" * 72)
for name, acc, f1 in summary_rows:
    print(f"{name:52s}{acc:10.3f}{f1:10.3f}")
print("-" * 72)
print(f"{'Literature — BloodMNIST VQC-EfficientNet (2025)':52s}{0.9658:10.3f}{'—':>10s}")
print()
print("Per Section 8's definition of success: the goal is comparability with auto-generated-")
print("ansatz prior work, not beating it outright — the contribution here is interpretability")
print(f"(N_TRAIN={N_TRAIN} images is a tiny fraction of BloodMNIST's 17,092-image benchmark, so")
print("read this as a feasibility/methodology comparison, not an apples-to-apples accuracy claim).")

## 14. What this confirms (and what's genuinely still open)

**Confirmed, end-to-end, on real labelled data:**
- PCA (8 components, train-fit-only) -> reshape (2x4) -> a deeper, **trainable** CNN ->
  differentiable [0, π] sigmoid-squash -> 8-qubit angle embedding -> built-in `n_local`
  ansatz with **RZZ entanglers** (38 trainable parameters: Ry rotations + parameterized
  ZZ-interaction entangling gates, per the professor's recommendation) -> Pauli-Z readout.
- A real **hybrid training loop**: a single Adam optimizer jointly updating the CNN's weights
  and the ansatz's parameters, with gradients flowing end-to-end through a custom
  parameter-shift autograd layer (verified against numerical finite differences) — a falling
  loss curve and a held-out test evaluation (accuracy + F1), not just one untrained forward
  pass.
- A **same-features classical baseline** (SVM), now trained on the *trained* CNN's output, for
  the apples-to-apples comparison Section 4 / Q3 says a reviewer will ask about.
- A **barren-plateau gradient-variance check** at initialization (Section 3.6 / Q10) — run,
  not just asserted.
- A **NISQ noise preview** via `FakeSherbrooke` (Section 3.5 / Section 9's Week-4 fallback),
  ahead of the real IBM hardware run.

**Deliberately still out of scope for this review, per the study guide:**
- The **manually constructed ansatz** (Section 3.2 / Objective 3) — the `n_local`/RZZ ansatz is
  still a Qiskit **built-in**. Hand-placing rotation gates and entanglers, and being able to
  justify each one individually, is the next review's deliverable.
- The **real IBM hardware run** (Week 4) — `FakeSherbrooke` is a noise-model preview, not a
  substitute for actual QPU time.
- Full-dataset training (`N_TRAIN`/`N_TEST` are small subsets for CPU-time reasons — see
  Section 10 of the guide; joint CNN+ansatz training roughly doubles the parameter-shift cost
  per step versus ansatz-only training, since the 8 encoding angles now need shifting too, and
  this ansatz's 38 parameters sit between `real_amplitudes`' 24 and `efficient_su2`'s 48 — see
  the companion notebooks for those CNOT-based alternatives).

**Next step (this is where the manual ansatz work begins):** replace the `n_local`/RZZ ansatz in
Section 6 with a hand-built circuit — parameterized single-qubit rotations you choose, plus
entangling-gate placement you choose (RZZ, CNOT, or otherwise) — and be ready to justify *why*
each gate is there, per Section 11's "fundamentals argument."
